# Smart Traveller Project.
# Using LangChain, VectorDB, RAG and an Agent Pipeline to help you plan your trip, grounded in truth.

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

# CREATING EMBEDDINGS & CHUNKING PROCESS

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# load the PDF's pages
PDF_PATH = "thailand_travel_guide.pdf"
loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Total number of pages loaded: {len(pages)}")

# to create chunks of the PDF
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 400,
    chunk_overlap = 100,
    separators= ['\n\n', '\n', '.', ' ']
)

chunks = splitter.split_documents(pages)
print(f"The number of chunks created are: {len(chunks)}")

C:\Users\Manim\AppData\Local\Temp\ipykernel_22488\2759402261.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Code\Python\Smart Traveller\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total number of pages loaded: 4
The number of chunks created are: 27


# STORE VECTORS IN VECTOR DB AND SET UP RETRIVAL FOR RAG

In [5]:
# to store embeddings in vector DB

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks, embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5386.63it/s]


# CONFIGURE LLM MODEL, TO ACT AS THE AGENT'S BRAIN
# Retrieve relevant chunks from Vector DB
# Create Chain to create a pipeline

# pass context & user query -> pass compiled prompt -> call LLM -> print content of AIMessage

In [17]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI

# define LLM model
model = ChatGoogleGenerativeAI(
    model = "gemini-3.6-flash",
    temperature = 0.25
)

# helper function
def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

SYSTEM_PROMPT = """You are an expert Smart Travel & Itinerary Planner. 
Provide clear, structured, day-by-day itineraries with practical tips on local food, culture, and budgeting.
Answer using only the context below:
{context}"""



# main method to define RAG pipeline, then invoke Model
def ask(query: str):
    # construct prompt using Messages API
    prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}")
])

    retriever = vector_store.as_retriever(search_kwargs={"k" : 5})
    retrieved = retriever.invoke(query)

    # create chain
    chain = (
        {"context" : retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | model
        | StrOutputParser()
    )
    print(f"Q: {test_query}")
    print(f"A: {chain.invoke(test_query)}")



In [18]:

test_query = "I have a 3-day budget trip planned for Bangkok starting from Bangalore. I care most about local street food and historic temples. What should my day-by-day plan look like?"
res = ask(test_query)

Q: I have a 3-day budget trip planned for Bangkok starting from Bangalore. I care most about local street food and historic temples. What should my day-by-day plan look like?


c:\Code\Python\Smart Traveller\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


A: Here is your structured, day-by-day itinerary based strictly on the provided details:

---

### **Day 1: Arrival, Currency Exchange, & River Cruise**
*   **Morning:** 
    *   Arrive at Bangkok Suvarnabhumi Airport (BKK) via Thai Airways TG 326 at 06:55 AM. 
    *   Clear customs and pick up an **AIS Tourist SIM card** (500 THB for 15 days unlimited 5G). 
    *   Take the Airport Rail Link to your hotel, **The Sukosol Bangkok**.
*   **Afternoon:** 
    *   Check-in and relax. 
    *   Head to **Siam Paragon** and **CentralWorld** for lunch.
    *   *Money Tip:* Exchange your currency at **SuperRich Exchange** located here for the best INR/THB rates.
*   **Evening:** 
    *   At 07:30 PM, board the **Opulence Chao Phraya Dinner Cruise** from ICONSIAM (Cost: 1,400 THB). 
    *   Enjoy a Thai seafood buffet, live music, and night views of the illuminated Grand Palace and Wat Arun.

---

### **Day 2: Historic Temples Tour**
*   **Morning (08:30 AM – 11:30 AM): The Grand Palace**
    *  